---
# <span style="color:blue">**MATRIX-BASED NUMERICAL SOLUTION of PDE** </span>

# Introduction
- The matrix method is introduced in the context of the **heat equation** (1D).
- This Notebook series includes four parts: Heat_Equation_PartA.ipynb,...,Heat_Equation_PartD.ipynb
- The goal of this series is to provide an introduction to solving PDEs using matrix methods.
- The purpose is to give Master of Science (Engineering) students the skills to independently carry out \
small-scale modeling
projects and the conceptual skills to move on to studying professional methods.
  

# Section topics:
## Part A: $ $ Finit difference method 
##  Part B: $ $ Implicit schemes & spectral analysis
##  Part C: $ $ Boundary conditions 
##  Part D: $ $ LU factorization and matrix exponentials

---

---

## Heat Equation Part A 

# <span style="color:blue">Finit difference method</span>

#### PDE → discretization → matrix operator 

---

## Learning objectives
By the end of this part, you will be able to:

- understand the physical meaning of the one-dimensional heat equation,
- discretize the spatial operator using finite differences,
- derive the discrete Laplacian matrix,
- visualize tridiagonal structure,
- interpret the matrix physically,
- express the resulting system as a matrix differential equation,
- implement explicit Euler,
- derive the stability condition,
- implement explicit time-stepping scheme.
    
## Contents
#### 1. Physical background
#### 2. System matrix formulation
#### 3. Finite Difference Approximation
   ##### &nbsp;&nbsp;&nbsp;&nbsp; by Second Derivative stencil        
#### 4. From Finite Differences to a Tridiagonal Matrix
#### 5. What Does This Matrix Do?
#### 6. Visualizing the system matrix (NumPy demo)
#### 7. Why is it called Explicit Euler?
#### 8. Stability condition for Explicit Euler
#### 9. Demonstration snippet about stability
#### 10. Implementation of rod cooling

---

👉 The heat equation models how **local temperature differences** drive global evolution over time.


## 1. Physical background

In this mini-project, **homogeneous Dirichlet boundary conditions** model a rod whose ends are held at zero temperature, allowing heat to leave the system. Other boundary conditions, such as insulated ends, would conserve total thermal energy and lead to redistribution rather than cooling.

The temperature $u(x,t)$ in a rod satisfies the **Heat Equation**:

$$
\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2},
$$
where $\alpha$ is a positive coefficient called the *thermal diffusivity* of the medium.

---

## 2. System matrix formulation

We will form a linear system of ODEs:

$$
\frac{d\mathbf{u}(t)}{dt} =   A \mathbf{u}(t),
$$

where system matrix $A$ represents the **discrete Laplacian operator** and is defined as follows:

$$
A =  \frac{\alpha}{h^2}L,
$$

where $L$ is the **tridiagonal Laplacian matrix** and $h$
 is the distance between adjacent nodes in the discretized spatial domain.

---

## 3. Finite Difference Approximation
   ### Second Derivative stencil  

To solve the 1D heat equation numerically, we **discretize the spatial domain into uniform grid points** $x_i = i h, \quad i = 0,1,\dots,N$.


The second derivative of a function $u(x)$ at point $x_i$ is approximated by the
[<ins>central finite difference formula</ins>](https://en.wikipedia.org/wiki/Finite_difference#Basic_types):


$$
\frac{\partial^2 u}{\partial x^2}(x_i)
\;\approx\;
\frac{u_{i+1} - 2u_i   + u_{i-1}}{h^2},
$$

where $u_i \approx u(x_i)$.


👉 Discretization **replaces derivatives with local differences**, turning the PDE into algebraic relations. Wikipedia:  <a href="https://en.wikipedia.org/wiki/Finite_difference_method#Finite_difference_method" target="_blank">
    <ins>Finite difference method</ins>.




## 4. From Finite Differences to a Tridiagonal Matrix

Applying this approximation at all *interior* grid points $ i = 1,\dots,N-1 $ produces a system of linear equations (assuming fixed boundary values at $x_0$ and $ x_N$ ). The vector $\mathbf{u}$ contains only the interior grid values; boundary values are fixed by the boundary conditions.


$$
\frac{1}{h^2}
\begin{bmatrix}
\textcolor{blue}{-2} & \textcolor{red}{1}  & 0  & \cdots & 0 \\
\textcolor{red}{1}  & \textcolor{blue}{-2} & \textcolor{red}{1}  & \ddots & \vdots \\
0  & \textcolor{red}{1}  & \textcolor{blue}{-2} & \ddots & 0 \\
\vdots & \ddots & \ddots & \ddots & \textcolor{red}{1} \\
0 & \cdots & 0 & \textcolor{red}{1} & \textcolor{blue}{-2}
\end{bmatrix}
\begin{bmatrix}
u_1 \\ u_2 \\ u_3 \\ \vdots \\ u_{N-1}
\end{bmatrix}.
$$


This matrix has:

- \(-2\) on the main diagonal (multiplies $u_i$)  
- \(1\) on the upper and lower diagonals  (multiplies and adds the neighbors of $u_i$ i.e.  $u_{i-1}$ and  $u_{i+1})$  
- zeros everywhere else  (only the temperatures of the neighbors are taken into account)

For example, when the second row of the matrix multiplies the vector **u**, we get 
$\frac{1}{h^2}(1u_1  - 2\\ u_2 + 1\\ u_3 )$,
which approximates the spatial second derivative $\partial^2 u / \partial x^2 $ at point $x_2$.

Such a matrix is called a **tridiagonal matrix**. The non-zero elements located only on the main diagonal, the subdiagonal (directly below), and the superdiagonal (directly above). All other entries are zero. This follows directly from the stencil structure $ \, 1u_{i-1}  - 2\\ u_i + 1\\ u_{i+1} $.

For interior points with spacing *h* we define matrix $A$ as follows:
$$
A = \frac{\alpha}{h^2} \begin{bmatrix} -2 & 1 & 0 & \cdots \\ 1 & -2 & 1 & \cdots \\ 0 & 1 & -2 & \cdots \\ \vdots & & & \ddots \end{bmatrix}
$$


---

## 5. What Does This Matrix Do?

This matrix is the **discrete Laplacian operator** in 1D. It:

- approximates the spatial second derivative $\partial^2 u / \partial x^2 $,
- encodes how each grid point interacts with its immediate neighbors,
- transforms the temperature vector $ \mathbf{u} $ into its **discrete curvature**.

In the heat equation

$$
\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2},
$$

this matrix controls how heat **diffuses** over space:

- large curvature → strong change in temperature,
- flat regions → little change.

From a linear algebra viewpoint, the heat equation becomes a matrix ODE:

$$
\frac{d\mathbf{u}}{dt} =  A \mathbf{u},
$$

where matrix $A$ approximates the second-derivative operator.

This is the key connection between **partial differential equations** and **matrix systems** used throughout this *Project_matrix*.


👉 Each update step depends only on neighboring values, reflecting physical diffusion.


## 6. Visualizing the system matrix A (NumPy demo)
For interior points with spacing *h*:
$$
A = \frac{\alpha}{h^2} \begin{bmatrix} -2 & 1 & 0 & \cdots \\ 1 & -2 & 1 & \cdots \\ 0 & 1 & -2 & \cdots \\ \vdots & & & \ddots \end{bmatrix}
$$

✅ The numerical scheme simulates how heat spreads through repeated local interactions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Construct the system matrix A 

N = 6                         # number of spatial grid points
alpha = 1.0                   # thermal diffusivity
h = 0.1                       # spatial step size

main = -2*np.ones(N)          # main diagonal
off  =  1*np.ones(N-1)        # off  diagonal 

A = (alpha/h**2)*(np.diag(main) + np.diag(off,1) + np.diag(off,-1))     

print(A)

---

## 7. Why is it called Explicit Euler? ###
We start from the 1D heat equation:
$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2} $$
After spatial discretization (finite differences):
$$\frac{d\mathbf{u}}{dt} = A \mathbf{u}$$
where: 
- $\mathbf{u}(t)$ is the vector of temperatures at grid points
- $A$ is the system matrix from the second derivative.

The time ODE system is:
$$\mathbf{u}'(t)=A\mathbf{u}(t) $$ 

### Euler method idea ###
Approximate the time derivative:
$$
\mathbf{u}'(t_n) \approx \frac{\mathbf{u}^{n+1} - \mathbf{u}^n}{\Delta t},
$$

where n denotes the timestep index. \
Plug in:
$$
\frac{\mathbf{u}^{n+1} - \mathbf{u}^n}{\Delta t} = A\mathbf{u}^n 
$$
Solve for next step:
$$
\mathbf{u}^{n+1} = \mathbf{u}^n + \Delta t\, A \mathbf{u}^n
$$
### Why “explicit”? ###
Because $\mathbf{u}^{n+1} $  is computed directly from $ \mathbf{u}^n $.
No equation solving. No matrix inversion. Just a matrix–vector multiplication.

---

## 8. Stability condition for Explicit Euler
For the heat equation, stability requires a sufficiently small size for the time step:

$$
\Delta t \le \frac{h^2}{2\alpha}
$$

This is the CFL (Courant–Friedrichs–Lewy) condition for diffusion.
If violated:
- solution oscillates
- then blows up exponentially

In Part B we will see that this condition follows directly from the spectrum of the matrix A.

---

## 9. Demonstration snippet about stability

👉 The matrix viewpoint reveals the structure behind the update rule.


In [ ]:
# Demonstration snippet about stability

h = 0.05                                   # spatial step size (h = Δx)
alpha = 1.0                                # thermal diffusivity 
               
dt_stable = h**2/(2*alpha)                 # stable time step size (Δt)
dt_unstable = 2*dt_stable                  # unstable time step size (Δt)

print("Stable Δt:  ", round(dt_stable, 5))
print("Unstable Δt:", round(dt_unstable, 5))

---

## 10. Implementation of rod cooling 

In [ ]:
# Initials

L = 1.0                            # rod length
alpha = 1.0                        # thermal diffusivity of the medium (α )
N = 50                             # number of spatial steps
T = 0.2                            # final time

x = np.linspace(0, L, N+2)         # x-axis discretization (rod coordinates)
h = L / (N+1)                      # spatial step size

# Boundary conditions: u(0,t)=u(L,t)=0 (implicit via interior grid only)
# Initial temperature distribution in the rod (sin shape) 
def initial_condition(x):
    return np.sin(np.pi * x)       # sin shape from 0 -> 180 deg,  0<x<1

u0 = initial_condition(x[1:-1])    # u0: NumPy array, (x[1:-1]): List x of lattice points on x axis (without the first and last member) 


In [ ]:
# Matrix A

main_diag = -2 * np.ones(N)        
off_diag = np.ones(N-1)            
A = (alpha / h**2) * (np.diag(main_diag) + np.diag(off_diag,1) + np.diag(off_diag,-1))   # Matrix:temperature distribution -> temp. curvature field



In [ ]:
# Optional prints 

#print(x)                          # lattice points in the rod
#print(h)                          # the length of one spatial step
print(1/(h*h))                    # see  matrix A
print(2/(h*h))                    # see  matrix A
#print(u0)                         # initial temperature distribution
#print(main_diag)
#print(off_diag)
print(A)                          # discrete Laplacian operator in 1D: approximates the spatial second derivative in spatial lattice points

In [ ]:
# Solve matrix ODEs

def solve_explicit(u0, A, dt, T):           # (initial temperature vector, system matrix A, time step dt, final time T)
    u = u0.copy()                                                             
    U_history = [u.copy()]                  
    #print(history)
    steps = int(np.round(T/dt))             # T=0.2,  dt=0.00015378...(in the next cell), --> number of time steps to reach final time T (= 1300)
    for _ in range(steps):                  # repeated 1300 times, ("throwaway variable" '_' a is required only syntactically)
        u = u + dt*A@u                      # du/dt is proportional to Au --> dtAu gives the list of temperature changes for the next time step
        U_history.append(u.copy())          # adding the new temperature distribution of the next time point to the U_history.
        
    #print("dt*A@u:", dt*A@u )              # temperature change computed from the current (already updated) state
    #print("u + dt*A@u:", u + dt*A@u )      # the temperature after one additional Euler step
    #print("np.array(U_history):", np.array(U_history)) 
    
    return np.array(U_history)              # 2D NumPy array: rows = time steps, columns = spatial grid points.

dt = 0.4 * h**2 / alpha                     # time step
U = solve_explicit(u0, A, dt, T)            # U_history of temperature distributions of all time moments during cooling (2D NumPy array)


print("time step dt =", round(dt, 7))
print("Final time =",round(1300*dt, 1))

In [ ]:
# Plott cooling process snapshots

# 2D NumPy array U has 1300 temperature distributions [U[0], U[1],...,U[1299], U[1300]] 

fig, ax = plt.subplots(figsize=(7, 4))

plt.plot(x[1:-1], U[0],   label='0, t=0')         #  U[0]: initial temperatures at spatial lattice points 
plt.plot(x[1:-1], U[325], label='325')   
plt.plot(x[1:-1], U[650], label='650')  
plt.plot(x[1:-1], U[975], label='975')       
plt.plot(x[1:-1], U[-1],  label='1300, t=0.2')    #  U[-1] = U[1300]: temperatures at time T = 1300*dt = 0.2  
plt.legend(); plt.grid();

ax.set_title("Cooling process snapshots")
ax.set_xlabel("x"); ax.set_ylabel("temperature"); ax.legend(title="time steps") 
ax.legend(); ax.grid() 

plt.show()

# print(U[650])                                      # Temperature distribution after 650 time steps

*Heikki Miettinen 2026*